# 01 · The tensile tests as data: two ontology generations, one neutral record

**Goal.** Show how the same tensile test is described in two generations of the Platform MaterialDigital core ontology, PMDco 2 and PMDco 3, both used together with the Tensile Test Ontology (TTO); how an *ontology profile* maps both descriptions to the same neutral records that the analysis needs; and how publication metadata for the Dataportal is harvested from the data's own semantics and then reviewed by a person.

**Audience.** Ontologists and RDF experts. MSE experts read it last on their path, to see where the analysed numbers come from.

**Prerequisites.**
- RDF and SPARQL basics.
- The `pmd-demonstrator` environment from `environment.yml`, which installs the package `pmd_tensile` from this repository (editable).
- `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN`, if the instance needs one) set in the environment before Jupyter starts. Neither is printed here. The Dataportal is only read; no Dataportal token is needed.

**Reading paths.** Ontologists: **01** → [04](04-tensile-workflow-knowledge-graph.ipynb) → [05](05-tensile-analysis-overview.ipynb). MSE experts: [05](05-tensile-analysis-overview.ipynb) → [02](02-tensile-analysis-nodes.ipynb) → **01**.

The ten tests on S355 steel (Zx1–Zx4 in rolling direction, Zy1–Zy4 perpendicular to it, Zd2 and Zd3 diagonal) are described in both generations. Zx3 is the example throughout.

| Section | Question |
| --- | --- |
| 1. Six datasets on Ontodocker | Which graphs are hosted, and how do the hosted PMDco 3 graphs relate to the committed files? |
| 2. Zx3 in both generations | How are S0, ReH and a curve column expressed? |
| 3. Ontology profiles | Where is the ontology-specific knowledge kept, and how is a graph checked against it? |
| 4. The same neutral records | Do both descriptions give the analysis the same input? |
| 5. The curve tables | How does each graph say where the raw curve is and what its columns mean? |
| 6. Metadata harvested from the graphs | What can the praeco harvester derive, and what must a person review? |
| 7. The Dataportal records | What was published, and does it match the review? |

*Execution class: network-read.* SPARQL queries and two Turtle downloads from Ontodocker, two curve downloads and one header read from the public data host, and public reads of two Dataportal records and one record file. Two committed files in `datasets/` are read for comparison. Nothing is written to remote services, the repository or the disk.

## Setup

The Ontodocker datasets are read with [praeco](https://github.com/materialdigital/praeco), either directly (praeco's `OntodockerClient`, built by `demo.ontodocker_client()` from the environment) or through `pmd_tensile.demo.select`, which runs a SPARQL SELECT on every dataset of a source. `short()` writes IRIs with the prefixes below; both generations use the same namespace `https://w3id.org/pmd/co/` (`pmdco:`), PMDco 2 with readable local names and PMDco 3 with numeric IDs.

In [ ]:
import dataclasses
import hashlib
import os
import re
import urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import rdflib
from IPython.display import display
from praeco import DataportalClient, Person, harvest_publication_metadata_from_rdf
from praeco.rdf_metadata import IncompleteHarvestError

import pmd_tensile
from pmd_tensile import acquisition, demo, exploration
from pmd_tensile.profiles import get_profile, pmdco2_tto, pmdco3_tto

pd.set_option("display.max_colwidth", None)  # pandas would otherwise cut IRIs, titles and evidence.

if not os.environ.get(demo.ONTODOCKER_ADDRESS_ENV, "").strip():
    raise RuntimeError("Set ONTODOCKER_ADDRESS (and ONTODOCKER_TOKEN if required) before starting Jupyter.")

ORIENTATIONS = ("parallel", "perpendicular", "diagonal")
GENERATIONS = {  # profile ID -> the three hosted datasets of that generation
    "pmdco2-tto": [f"pmdco2_tto_example_{orientation}" for orientation in ORIENTATIONS],
    "pmdco3-tto": [f"pmdco3_tto_example_{orientation}" for orientation in ORIENTATIONS],
}
SOURCES = {profile_id: {"kind": "ontodocker", "datasets": datasets} for profile_id, datasets in GENERATIONS.items()}
SPECIMEN = "Zx3"

# Committed files in datasets/, found through the editable install of pmd_tensile from this repository.
DATASETS_DIR = Path(pmd_tensile.__file__).resolve().parents[1] / "datasets"
if not (DATASETS_DIR / "pmdco3_tto_example.ttl").is_file():
    raise RuntimeError("datasets/ not found: install pmd_tensile editable from this repository (environment.yml).")

PREFIXES = {
    "pmdco": "https://w3id.org/pmd/co/",
    "tto": "https://w3id.org/pmd/tto/",
    "obo": "http://purl.obolibrary.org/obo/",
    "tte": "https://w3id.org/pmd/ao/tte/",  # PMDco 2 data
    "abox": "https://w3id.org/pmd/demodata/tensiletest_S355/",  # PMDco 3 data
    "csvw": "http://www.w3.org/ns/csvw#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "unit": "http://qudt.org/vocab/unit/",
    "orcid": "https://orcid.org/",
}
SPARQL_PREFIXES = "\n".join(f"PREFIX {prefix}: <{namespace}>" for prefix, namespace in PREFIXES.items())
# Labels of the PMDco 3/TTO/OBO terms from the ontology files at TTO commit 08a7aec, kept by the profile.
LABELS = {curie: label for curie, label, _ in pmdco3_tto.IRI_LABELS}


def short(text) -> str:
    """An IRI as prefix:name; anything else unchanged."""
    text = "" if text is None else str(text)
    for prefix, namespace in PREFIXES.items():
        if text.startswith(namespace):
            return f"{prefix}:{text[len(namespace):]}"
    return text


def short_n3(text: str) -> str:
    """Shorten every <IRI> in an N3 term or triple."""
    return re.sub(r"<([^>]+)>", lambda match: short(match.group(1)), text)


def select(dataset: str, query: str, columns: list[str]) -> pd.DataFrame:
    """SPARQL SELECT on one hosted dataset (the package's Ontodocker access)."""
    rows = demo.select({"kind": "ontodocker", "datasets": [dataset]}, SPARQL_PREFIXES + "\n" + query, columns)
    return rows.drop(columns="source_part")


print("Profiles:", ", ".join(GENERATIONS))
print("Example specimen:", SPECIMEN)

## 1. Six datasets on Ontodocker

Ontodocker hosts each orientation group of the tests as one dataset, in both generations. The PMDco 2 datasets were already hosted; the PMDco 3 ones were cut by this demonstrator from the original PMDco 3 graph of the TTO repository (commit `08a7aec`) into the same three groups, with per-test curve tables added (section 5), and each title extended by its orientation (`scripts/slice_pmdco3.py`).

The table lists the six datasets of the two generations, whether Ontodocker hosts each, and the number of tensile tests in each (class `pmdco:TensileTest` in PMDco 2, `pmdco:PMD_0000974`, "tensile testing process", in PMDco 3) and the title of the ontology header.

In [ ]:
ontodocker = demo.ontodocker_client()  # praeco's OntodockerClient; the package reads address and token
retrieved = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
available = set(ontodocker.datasets.list())

TESTS_AND_TITLE = """
SELECT ?tests ?title WHERE {
  { SELECT (COUNT(DISTINCT ?test) AS ?tests) WHERE { VALUES ?class { pmdco:TensileTest pmdco:PMD_0000974 } ?test a ?class } }
  OPTIONAL { ?ontology a owl:Ontology ; dc:title ?title }
}
"""
overview = []
for profile_id, names in GENERATIONS.items():
    for name in names:
        row = {"dataset": name, "generation": "PMDco 2" if profile_id == "pmdco2-tto" else "PMDco 3",
               "hosted": name in available, "tensile tests": None, "dc:title": "–"}
        if row["hosted"]:
            answer = select(name, TESTS_AND_TITLE, ["tests", "title"]).iloc[0]
            row.update({"tensile tests": int(answer["tests"]), "dc:title": answer["title"]})
        overview.append(row)
display(pd.DataFrame(overview).set_index("dataset"))
print("Data retrieved:", retrieved)

### Hosted and committed PMDco 3 graphs

The slices are committed in `datasets/` and were uploaded unchanged. Ontodocker returns them with one systematic difference: literals written as `"…"^^xsd:string` come back as plain literals `"…"`. Under RDF 1.1 these are the same term, but rdflib 7 keeps them apart, so a strict comparison in rdflib reports a difference. The comparison below therefore maps `xsd:string` to plain literals first. Blank nodes (the RDF lists of the curve-table columns) are only counted, because a full isomorphism check is slow on graphs with many blank nodes.

In [ ]:
XSD_STRING = rdflib.XSD.string


def plain_strings(graph: rdflib.Graph) -> set:
    """Triples without blank nodes, with xsd:string literals written as plain literals."""
    mapped = set()
    for s, p, o in graph:
        if isinstance(s, rdflib.BNode) or isinstance(o, rdflib.BNode):
            continue
        if isinstance(o, rdflib.Literal) and o.datatype == XSD_STRING:
            o = rdflib.Literal(str(o))
        mapped.add((s, p, o))
    return mapped


def count(graph: rdflib.Graph, condition) -> int:
    return sum(1 for triple in graph if condition(triple))


pmdco3_parallel_text = ontodocker.datasets.fetch_turtle("pmdco3_tto_example_parallel")  # reused in section 6
graphs = {
    "hosted (Ontodocker)": rdflib.Graph().parse(data=pmdco3_parallel_text, format="turtle"),
    "committed (datasets/pmdco3_tto_example_parallel.ttl)": rdflib.Graph().parse(
        DATASETS_DIR / "pmdco3_tto_example_parallel.ttl", format="turtle"
    ),
}
display(pd.DataFrame({
    name: {
        "triples": len(graph),
        "literals typed xsd:string": count(graph, lambda t: isinstance(t[2], rdflib.Literal) and t[2].datatype == XSD_STRING),
        "triples with a blank node": count(graph, lambda t: isinstance(t[0], rdflib.BNode) or isinstance(t[2], rdflib.BNode)),
    }
    for name, graph in graphs.items()
}))
hosted_graph, committed_graph = graphs.values()
print("Triples without blank nodes equal after mapping xsd:string to plain literals:",
      plain_strings(hosted_graph) == plain_strings(committed_graph))

## 2. Zx3 in both generations

Three things the analysis needs, for one specimen: the original cross-section S0, the lab-reported upper yield strength ReH, and the force column of the raw curve. One small query per generation finds them for the test piece named `Zx3`, in the dataset of its orientation group.

In [ ]:
ZX3_PMDCO2 = """
SELECT ?quantity ?node ?class ?value ?unit (DATATYPE(?value) AS ?datatype) WHERE {
  ?test a pmdco:TensileTest ;
        pmdco:input/pmdco:characteristic ?name .
  ?name a pmdco:TestPieceName ; pmdco:value "Zx3" .
  {
    ?test pmdco:characteristic ?node .
    ?node a pmdco:CrossSectionArea ; pmdco:value ?value ; pmdco:unit ?unit .
    FILTER(STRENDS(STR(?node), "_S0"))  # S0, Su and s1..s3 share the class; only the IRI tells them apart
    BIND("S0" AS ?quantity) BIND(pmdco:CrossSectionArea AS ?class)
  } UNION {
    ?test pmdco:characteristic ?node .
    ?node a pmdco:UpperYieldStrength ; pmdco:value ?value ; pmdco:unit ?unit .
    BIND("ReH" AS ?quantity) BIND(pmdco:UpperYieldStrength AS ?class)
  } UNION {
    ?test pmdco:characteristic/pmdco:resource/csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?node .
    ?node a pmdco:Force ; pmdco:unit ?unit .
    BIND("force column" AS ?quantity) BIND(pmdco:Force AS ?class)
  }
}
"""

ZX3_PMDCO3 = """
SELECT ?quantity ?node ?class ?value ?unit (DATATYPE(?value) AS ?datatype) WHERE {
  ?test a pmdco:PMD_0000974 ;                                # tensile testing process
        obo:OBI_0000293 ?test_piece .                        # has specified input
  ?identifier obo:IAO_0000219 ?test_piece ;                  # denotes
              obo:OBI_0001938/pmdco:PMD_0000006 "Zx3" .      # has value specification / has value
  {
    ?test_piece obo:RO_0000086 ?node .                       # has quality
    ?node a tto:TTO_0000026 .
    BIND("S0" AS ?quantity) BIND(tto:TTO_0000026 AS ?class)
  } UNION {
    ?test_piece pmdco:PMD_0020005/obo:RO_0000086 ?node .     # test piece -> material ("porosity" in the ontology files), has quality
    ?node a tto:TTO_0000059 .
    BIND("ReH" AS ?quantity) BIND(tto:TTO_0000059 AS ?class)
  } UNION {
    ?test obo:OBI_0000299 ?table .                           # has specified output
    ?table a pmdco:PMD_0000992 ;                             # time series
           csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?node .
    ?node obo:IAO_0000136 ?entity ;                          # is about
          obo:IAO_0000039 ?unit .                            # has measurement unit label
    ?entity a pmdco:PMD_0020200 .
    BIND("force column" AS ?quantity) BIND(pmdco:PMD_0020200 AS ?class)
  }
  OPTIONAL {
    ?test obo:OBI_0000299 ?datum .                           # has specified output
    ?datum a obo:IAO_0000109 ;                               # measurement datum
           obo:IAO_0000136 ?node ;                           # is about
           obo:OBI_0001938 ?spec .                           # has value specification
    ?spec pmdco:PMD_0000006 ?value ;                         # has value
          obo:IAO_0000039 ?unit .                            # has measurement unit label
  }
}
"""

COLUMNS = ["quantity", "node", "class", "value", "unit", "datatype"]
side_by_side = []
for profile_id, query, dataset in (
    ("pmdco2-tto", ZX3_PMDCO2, "pmdco2_tto_example_parallel"),
    ("pmdco3-tto", ZX3_PMDCO3, "pmdco3_tto_example_parallel"),
):
    profile = get_profile(profile_id)
    for row in select(dataset, query, COLUMNS).to_dict("records"):
        side_by_side.append({
            "quantity": row["quantity"],
            "generation": "PMDco 2" if profile_id == "pmdco2-tto" else "PMDco 3",
            "node": short(row["node"]) if str(row["node"]).startswith("http") else "(blank node)",
            "class": short(row["class"]),
            "label of the class": LABELS.get(short(row["class"]), "–"),
            "value": row["value"] if row["value"] is not None else "–",
            "datatype": short(row["datatype"]) if row["datatype"] else "–",
            "unit": short(row["unit"]),
            "unit for the analysis": profile.unit_symbol(row["unit"]),
        })
order = {"S0": 0, "ReH": 1, "force column": 2}
side_by_side = pd.DataFrame(side_by_side)
display(side_by_side.assign(order=side_by_side["quantity"].map(order))
        .sort_values(["order", "generation"]).drop(columns="order").set_index(["quantity", "generation"]))

The two generations say the same about Zx3, but in different shapes:

- **PMDco 2** hangs each value directly on a node that is a `pmdco:characteristic` of the test, with `pmdco:value` and `pmdco:unit`; the numbers are plain string literals. S0, Su and s1–s3 share the class `pmdco:CrossSectionArea`, so only the local name of the IRI (`…_crossSectionArea_S0`) says which one is S0. The force column of the curve table is a blank node typed directly with the quantity class `pmdco:Force`.
- **PMDco 3** separates the *quality* (S0 is a quality of the test piece, ReH a quality of its material; the data reach the material through `pmdco:PMD_0020005`, which the pinned ontology files label *porosity*, an `owl:Class`) from the *measurement datum* that is about it. The number, typed `xsd:float`, and the unit sit on the datum's value specification. S0 has its own class. Classes are numeric IDs; their labels are not in the data graphs and come from the ontology files (here from the profile's table `pmdco3_tto.IRI_LABELS`, taken from TTO at commit `08a7aec`). In the curve tables that this demonstrator added (section 5), the force column is itself a measurement datum that is about a force.
- Both use the unit IRI `unit:MegaPa` for ReH, which is not a QUDT IRI (QUDT writes `unit:MegaPA`). The profiles map it to MPa by a declared alias; any other unknown unit raises an error.

## 3. Ontology profiles

The analysis in notebooks 02–05 never sees an IRI. Everything ontology-specific is in an *ontology profile*, a declarative object in `pmd_tensile.profiles` that holds SPARQL text and conventions, but no graph-access code:

- **conformance ASKs**, checked before anything is read;
- SELECT queries with fixed result columns for the **specimens** (test, test-piece name, orientation), the **quantities** (S0, s1–s3, L0, Lu, Su, the test-speed switch point, and the lab-reported values), the **notes**, and the **curve columns** (curve URL, column position, role and unit);
- **unit aliases** for non-QUDT unit IRIs found in the data.

Supporting another ontology means writing another profile; the exploration and acquisition code is shared.

In [ ]:
display(pd.DataFrame([
    {
        "profile": profile.profile_id,
        "description": profile.description,
        "conformance ASKs": len(profile.conformance_asks),
        "SELECT queries": ", ".join(
            name for name in ("specimens", "quantities", "notes", "curve_columns") if getattr(profile, name) is not None
        ),
        "unit aliases": "; ".join(f"{short(alias)} → {short(canonical)}" for alias, canonical in profile.unit_aliases.items()),
    }
    for profile in map(get_profile, GENERATIONS)
]).set_index("profile"))

### Conformance checks

Each profile's ASK queries must all hold in every dataset of a source before the workflow lists any specimen. Run on its own generation, every ASK holds. Run on the other generation, none does, and the workflow's first node, `check_conformance`, stops with an error.

In [ ]:
for profile_id, source in SOURCES.items():
    print(f"{profile_id} on its own datasets:")
    display(pd.DataFrame(demo.conformance_results(profile_id, source)).T)

for profile_id, other in (("pmdco2-tto", "pmdco3_tto_example_parallel"), ("pmdco3-tto", "pmdco2_tto_example_parallel")):
    answers = demo.conformance_results(profile_id, {"kind": "ontodocker", "datasets": [other]})
    held = sum(answer for by_dataset in answers.values() for answer in by_dataset.values())
    print(f"{profile_id} on {other}: {held} of {len(answers)} ASKs hold")

try:
    exploration.check_conformance("pmdco3-tto", {"kind": "ontodocker", "datasets": ["pmdco2_tto_example_parallel"]})
except demo.ConformanceError as error:
    print(f"check_conformance: {type(error).__name__}: {error}")

## 4. The same neutral records

The workflow's first steps are the exploration nodes (`check_conformance`, `list_specimens`) and the acquisition nodes (`get_specimen_record`, `get_reported_properties`, `fetch_curve`). They are the only nodes that know an ontology, and they take the profile ID and the source as plain inputs. A node is a plain Python function, so here they are called directly, once per generation.

Their outputs are the neutral records of `pmd_tensile.contract`, in fixed units (N, mm, mm², strain as a ratio, MPa, GPa, %). Every unit is converted through an explicit table.

In [ ]:
listed, tables = {}, {}
for profile_id, source in SOURCES.items():
    checked = exploration.check_conformance(profile_id, source)  # raises if an ASK fails; returns the profile ID
    listed[profile_id] = exploration.list_specimens(checked, source)
    tables[profile_id] = demo.specimen_table(profile_id, source).set_index("specimen_id")

pmdco2_table, pmdco3_table = tables["pmdco2-tto"], tables["pmdco3-tto"]
display(pd.DataFrame({
    "orientation (PMDco 2)": pmdco2_table["orientation"],
    "orientation (PMDco 3)": pmdco3_table["orientation"],
    "test (PMDco 2)": pmdco2_table["process"].map(short),
    "test (PMDco 3)": pmdco3_table["process"].map(short),
}))
print("Same specimen IDs:", listed["pmdco2-tto"] == listed["pmdco3-tto"])
print("Same orientation for every specimen:", pmdco2_table["orientation"].equals(pmdco3_table["orientation"]))

For Zx3, the specimen record (geometry, orientation, lab notes, curve location and column semantics), the lab-reported values and the raw curve, from each generation:

In [ ]:
records, reported, curves = {}, {}, {}
for profile_id, source in SOURCES.items():
    records[profile_id] = acquisition.get_specimen_record(profile_id, source, SPECIMEN)
    reported[profile_id] = acquisition.get_reported_properties(profile_id, source, SPECIMEN)
    curves[profile_id] = acquisition.fetch_curve(records[profile_id])


def columns_text(columns) -> str:
    return "; ".join(f"{role}: column {spec.index + 1}, {spec.unit}" for role, spec in columns.items())


def side_by_side_fields(objects: dict) -> pd.DataFrame:
    rows = []
    for field in dataclasses.fields(objects["pmdco2-tto"]):
        values = [getattr(objects[profile_id], field.name) for profile_id in GENERATIONS]
        shown = [columns_text(value) if field.name == "curve_columns" else value for value in values]
        rows.append({"field": field.name, "PMDco 2": shown[0], "PMDco 3": shown[1], "equal": values[0] == values[1]})
    return pd.DataFrame(rows).set_index("field")


record_table = side_by_side_fields(records)
reported_table = side_by_side_fields(reported)
display(record_table)
display(reported_table)

pmdco2_curve, pmdco3_curve = curves["pmdco2-tto"], curves["pmdco3-tto"]
display(pd.DataFrame({
    channel: {
        "samples": len(getattr(pmdco2_curve, channel)),
        "identical arrays": np.array_equal(getattr(pmdco2_curve, channel), getattr(pmdco3_curve, channel), equal_nan=True),
    }
    for channel in ("time_s", "crosshead_mm", "force_n", "strain")
}).T)

differing = list(record_table.index[~record_table["equal"]]) + list(reported_table.index[~reported_table["equal"]])
print(f"Fields compared: {len(record_table)} of the specimen record, {len(reported_table)} of the lab-reported values.")
print("Fields that differ:", ", ".join(differing) or "none")
print("Same curve bytes (SHA-256):", pmdco2_curve.sha256 == pmdco3_curve.sha256, pmdco2_curve.sha256)

Every field of the two records is equal except `provenance_iri`, the IRI of the test in its graph, which the record keeps for reporting only and no analysis step reads. ReL is empty in both, because neither graph gives a value for Zx3. Both records point to the same curve file on the data host (by construction: the added PMDco 3 curve tables link the files that PMDco 2 links, section 5), and the downloaded bytes are identical. From here on, the analysis cannot tell which generation the data came from; notebook 05 runs it on all ten specimens from both and compares the results.

## 5. The curve tables

A record is complete only with its raw curve: where the CSV file is, which column is time, crosshead travel, force and strain, and in which unit. Both generations say this with [CSVW](https://www.w3.org/TR/tabular-metadata/) tables, which each profile reads with its `CURVE_COLUMNS` query. The rows for Zx3:

In [ ]:
curve_rows = {}
for profile_id, module, dataset in (
    ("pmdco2-tto", pmdco2_tto, "pmdco2_tto_example_parallel"),
    ("pmdco3-tto", pmdco3_tto, "pmdco3_tto_example_parallel"),
):
    query = module.CURVE_COLUMNS  # the profile's own query, with its own prefixes
    rows = demo.select({"kind": "ontodocker", "datasets": [dataset]}, query.query, list(query.columns))
    rows = rows[rows["process"] == records[profile_id].provenance_iri].drop(columns=["process", "source_part"])
    rows["unit"] = rows["unit"].map(short)
    curve_rows[profile_id] = rows.sort_values("column_number").set_index("role")
    print(f"{profile_id} ({dataset}):")
    display(curve_rows[profile_id])

The PMDco 2 graph declares no CSV dialect (no `csvw:Dialect`), nor does the original PMDco 3 graph. The curve tables that this demonstrator added to the PMDco 3 slices declare one (`;` as delimiter, two header rows, `,` as decimal mark), and acquisition checks the declaration against the layout its CSV parser reads (not shown here; `tests/test_pmdco3_curve_tables.py` covers a declaration the parser cannot read). The column names in the graphs, compared with the header rows of the file itself:

In [ ]:
NAMES_PMDCO2 = """
SELECT ?role ?name ?title WHERE {
  VALUES (?class ?role) { (pmdco:Duration "time") (pmdco:CrossheadSeparation "crosshead") (pmdco:Force "force") (pmdco:PercentageExtension "strain") }
  ?table csvw:url ?url ; csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?column .
  FILTER(STR(?url) = "%s")
  ?column a ?class ; csvw:name ?name ; dc:title ?title .
}
"""
NAMES_PMDCO3 = """
SELECT ?role ?name ?title WHERE {
  VALUES (?class ?role) { (obo:BFO_0000202 "time") (tto:TTO_0000013 "crosshead") (pmdco:PMD_0020200 "force") (pmdco:PMD_0060011 "strain") }
  ?table a pmdco:PMD_0000992 ; csvw:url ?url ; csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?column .
  FILTER(STR(?url) = "%s")
  ?column obo:IAO_0000136/rdf:type ?class ; csvw:name ?name ; csvw:title ?title .
}
"""
url = records["pmdco3-tto"].curve_url
names = {
    "PMDco 2": select("pmdco2_tto_example_parallel", NAMES_PMDCO2 % url, ["role", "name", "title"]).set_index("role"),
    "PMDco 3": select("pmdco3_tto_example_parallel", NAMES_PMDCO3 % url, ["role", "name", "title"]).set_index("role"),
}
with urllib.request.urlopen(url, timeout=30) as response:  # token-free GET of the first bytes only
    header_lines = response.read(200).decode("utf-8").splitlines()[:2]
header_names, header_units = (line.split(";") for line in header_lines)

roles = list(curve_rows["pmdco3-tto"].index)
positions = [int(curve_rows["pmdco3-tto"].loc[role, "column_number"]) for role in roles]
display(pd.DataFrame({
    "column": positions,
    "PMDco 2 csvw:name": [names["PMDco 2"].loc[role, "name"] for role in roles],
    "PMDco 2 dc:title": [names["PMDco 2"].loc[role, "title"] for role in roles],
    "PMDco 3 csvw:title": [names["PMDco 3"].loc[role, "title"] for role in roles],
    "PMDco 3 csvw:name": [names["PMDco 3"].loc[role, "name"] for role in roles],
    "CSV header row 1": [header_names[position - 1] for position in positions],
    "CSV header row 2": [header_units[position - 1] for position in positions],
}, index=pd.Index(roles, name="role")))

In the added PMDco 3 curve tables, the titles repeat the CSV headers exactly; `csvw:name` is a sanitised identifier. The PMDco 2 names and titles differ from the headers. Both profiles locate each column by its position in the `csvw:column` list, not by name, and the CSV parser checks the unit row of the file against the unit the graph declares (not shown here; `tests/test_acquisition_curves.py` covers a mismatch).

### The PMDco 3 curve tables are additions

The original PMDco 3 graph in the TTO repository (commit `08a7aec`) links only one aggregate table, the lab's `S355_data.csv` with 57 columns of metadata and results for all ten tests, but none of the per-test curve files. This demonstrator added one curve table per test, with a modelling approved by the demonstrator's maintainer (the TTO maintainers have not reviewed it):

- the table is a `csvw:Table` and a `pmdco:PMD_0000992` (time series), linked from the test by `obo:OBI_0000299` (has specified output), with `csvw:url` pointing to the same CSV file that PMDco 2 links;
- each column is a `csvw:Column` and an `obo:IAO_0000109` (measurement datum) with its unit (`obo:IAO_0000039`), and is about (`obo:IAO_0000136`) a per-test entity typed with the quantity class: a temporal interval (`obo:BFO_0000202`) that the testing step occupies, the crosshead separation of the machine (`tto:TTO_0000013`), and the force (`pmdco:PMD_0020200`) and percentage extension (`pmdco:PMD_0060011`) of the test piece;
- the CSV layout is a named `csvw:Dialect` and a `csvw:NumericFormat`; the column order is an RDF list.

Each added table carries an `rdfs:comment` that says so, and the lab table carries one for its new URL on the data host. Readers of the graph can therefore tell the additions from the original:

In [ ]:
MARKED = """
SELECT ?table ?url ?comment WHERE {
  ?table a csvw:Table ; csvw:url ?url ; rdfs:comment ?comment .
  FILTER(CONTAINS(STR(?comment), "PMD demonstrator"))
}
"""
marked = pd.concat(
    [select(dataset, MARKED, ["table", "url", "comment"]).assign(dataset=dataset) for dataset in GENERATIONS["pmdco3-tto"]],
    ignore_index=True,
)
marked["table"] = marked["table"].map(short)
display(marked.groupby("comment")["table"].agg(
    lambda tables: f"distinct tables: {len(set(tables))}, e.g. {sorted(set(tables))[0]}"
).to_frame())
print("Curve tables marked as added, per dataset:",
      marked[marked["comment"].str.startswith("Added")].groupby("dataset")["table"].nunique().to_dict())

## 6. Metadata harvested from the graphs

A Dataportal record needs a title, a description, creators, a license, keywords and a publication date. The praeco 0.2.0 harvester (`harvest_publication_metadata_from_rdf`) reads these from an RDF graph, with the triples it used as evidence. It distinguishes three kinds of result:

- a **candidate** states the field directly (e.g. `dc:title` on the ontology header) and is resolved automatically;
- a **suggestion** is derived from the data (e.g. a description built from the tests, material and standard it finds); it must be accepted explicitly;
- a **review item** (diagnostic) blocks the conversion into publication metadata until a person resolves it.

It knows the patterns of both generations. Below, it runs on the hosted parallel slice of each, fetched as Turtle from Ontodocker.

In [ ]:
harvest_texts = {
    "pmdco2_tto_example_parallel": ontodocker.datasets.fetch_turtle("pmdco2_tto_example_parallel"),
    "pmdco3_tto_example_parallel": pmdco3_parallel_text,  # fetched in section 1
}
harvests = {}
for name, text in harvest_texts.items():
    harvest = harvest_publication_metadata_from_rdf(text)
    eligible = [short(candidate.subject.term) for candidate in harvest.subject_candidates if candidate.eligible]
    print(f"{name}: {len(harvest.subjects)} subjects inspected; eligible as publication subject: {', '.join(eligible)}")
    harvests[name] = harvest.select_subject_automatically()

FIELDS = ("title", "description", "keywords", "publication_date", "license")


def shown(value) -> str:
    if isinstance(value, tuple):
        return ", ".join(map(str, value))
    return "–" if value is None else str(value)


rows = []
for name, harvest in harvests.items():
    for field in FIELDS:
        review = getattr(harvest, field)
        rows.append({
            "dataset": name,
            "field": field,
            "status": review.status,
            "candidate": "; ".join(shown(item.value) for item in review.candidates) or "–",
            "suggestion": "; ".join(shown(item.value) for item in review.suggestions) or "–",
            "evidence rules": ", ".join(
                sorted({evidence.rule for item in review.candidates + review.suggestions for evidence in item.evidence})
            ) or "–",
        })
display(pd.DataFrame(rows).set_index(["dataset", "field"]))

The evidence of a suggestion lists the triples it rests on. For the description, the harvester found the tensile tests, the material and the test standard through different patterns in each generation (first two triples per rule):

In [ ]:
rows = []
for name, harvest in harvests.items():
    for suggestion in harvest.description.suggestions:
        for evidence in suggestion.evidence:
            for triple in evidence.triples[:2]:
                rows.append({
                    "dataset": name,
                    "rule": f"{evidence.rule} ({len(evidence.triples)} triples)",
                    "triple": " ".join(short_n3(term) for term in triple),
                })
display(pd.DataFrame(rows).set_index(["dataset", "rule"]))

What the harvester could not resolve, as creators it observed and as blocking review items:

In [ ]:
for name, harvest in harvests.items():
    observations = harvest.creators.observations
    print(f"{name}: creators {harvest.creators.status}, {len(observations)} observed")
    if observations:
        display(pd.DataFrame([
            {"creator": short(o.term), "kind": o.kind, "name": o.name or "–", "relations": ", ".join(o.relations)}
            for o in observations
        ]).set_index("creator"))
    display(pd.DataFrame([
        {"code": d.code, "field": d.field, "blocking": d.blocking, "message": d.message} for d in harvest.diagnostics
    ]))

The PMDco 2 graph names four creators by ORCID IRI, but the harvester cannot resolve any of them to a person. The header shows why: all four names are labels of one ORCID IRI, and the other three IRIs have no label, so the graph does not say which name belongs to which ORCID. The PMDco 3 graph names no creator at all, and neither graph states a license.

In [ ]:
CREATORS = """
SELECT ?creator (COUNT(?label) AS ?labels) (GROUP_CONCAT(?label; separator=", ") AS ?names) WHERE {
  ?ontology a owl:Ontology ; dc:creator ?creator .
  OPTIONAL { ?creator rdfs:label ?label }
} GROUP BY ?creator ORDER BY ?creator
"""
creators = select("pmdco2_tto_example_parallel", CREATORS, ["creator", "labels", "names"])
creators["creator"] = creators["creator"].map(short)
display(creators.set_index("creator"))

### The review step, as done for the PMDco 3 record

The Dataportal record `pmdco3-tto-recombined` was built from the harvest of the full PMDco 3 file `datasets/pmdco3_tto_example.ttl` (all ten tests; the record's Turtle artifact), reviewed as follows. The harvester refuses to produce publication metadata before the review:

In [ ]:
artifact = DATASETS_DIR / "pmdco3_tto_example.ttl"
harvest = harvest_publication_metadata_from_rdf(artifact).select_subject_automatically()
print("Description suggestion:", harvest.description.suggestions[0].value)
print("Publication date suggestion:", harvest.publication_date.suggestions[0].value, "(dcterms:created of the graph)")
try:
    harvest.to_publication_metadata()
except IncompleteHarvestError as error:
    print("Not yet publishable:", error)

The reviewer's decisions:

- **accept** the suggested description, which is derived from the graph;
- **override** the title, to mirror the PMDco 2 record's title with "(PMDco 3)" appended;
- **supply** the creators: the same four people as on the PMDco 2 record, without ORCIDs, because the PMDco 2 graph leaves the pairing open;
- **supply** the license (CC-BY-4.0, as for PMDco 2), which no graph states;
- **supply** the keywords: the PMDco 2 record's tags plus "PMDco 3" and "TTO" (they include the suggested "S355" and "tensile test");
- **supply** the publication date: the day of publication, not the graph's creation date.

In [ ]:
reviewed = harvest.accept_suggestion(harvest.description.suggestions[0]).with_overrides(
    title="Recombined S355 tensile test RDF dataset (PMDco 3)",
    creators=tuple(
        Person(name=name)
        for name in ("Bernd Bayerlein", "Jörg Waitelonis", "Markus Schilling", "Philipp von Hartrott")
    ),
    license="CC-BY-4.0",
    keywords=("Platform MaterialDigital", "PMD", "PMDco 3", "TTO", "S355", "tensile test", "RDF", "Turtle", "CSVW", "Ontodocker"),
    publication_date=date(2026, 11, 9),
)
metadata = reviewed.to_publication_metadata()

display(pd.DataFrame(
    [{"field": field, "status": getattr(reviewed, field).status, "origin": getattr(reviewed, field).origin,
      "value": shown(getattr(metadata, field))} for field in FIELDS]
    + [{"field": "creators", "status": reviewed.creators.status, "origin": reviewed.creators.origin,
        "value": ", ".join(person.name for person in metadata.creators)}]
).set_index("field"))

`origin` records how each field was resolved: `acceptance` of a suggestion derived from the graph, or `override` by the reviewer. Only the description comes from the graph here; the title had a candidate from `dc:title` that the reviewer replaced.

## 7. The Dataportal records

Both records are public and are read here without a token: `pmdco2-tto-recombined`, published earlier by `demonstrator.ipynb` with metadata entered in that notebook, and `pmdco3-tto-recombined`, published by this demonstrator from the reviewed harvest above. Each holds its combined Turtle file, links to the CSV files the graph references, and further resources that the Dataportal derived from these links; the demonstrator uploaded only the Turtle file and created the links.

In [ ]:
dataportal = DataportalClient()  # public read, no token
published = {name: dataportal.datasets.show(name).raw for name in ("pmdco2-tto-recombined", "pmdco3-tto-recombined")}
extras = {name: {item["key"]: item["value"] for item in record.get("extras", [])} for name, record in published.items()}

display(pd.DataFrame({
    name: {
        "title": record["title"],
        "license": record["license_id"],
        "public": not record["private"],
        "organization": (record.get("organization") or {}).get("name"),
        "creators": ", ".join(creator["name"] for creator in record.get("creator") or []),
        "tags": ", ".join(sorted(tag["name"] for tag in record["tags"])),
        "issued": record.get("issued"),
        "description": record["notes"],
        "harvested_by (extra)": extras[name].get("harvested_by", "–"),
        "resources by format": ", ".join(
            f"{fmt}: {n}" for fmt, n in pd.Series([r["format"] for r in record["resources"]]).value_counts().items()
        ),
        "page": f"{dataportal.base_url}/dataset/{name}",
    }
    for name, record in published.items()
}))

The PMDco 3 record against the reviewed metadata, its CSV links against the `csvw:url` values of the committed file, and its Turtle artifact (downloaded from the record) against the committed file:

In [ ]:
record = published["pmdco3-tto-recombined"]
pmdco2_tags = {tag["name"] for tag in published["pmdco2-tto-recombined"]["tags"]}
artifact_graph = rdflib.Graph().parse(artifact, format="turtle")
graph_csv_urls = sorted({str(url) for url in artifact_graph.objects(None, rdflib.URIRef(PREFIXES["csvw"] + "url"))})
record_csv_urls = sorted(r["url"] for r in record["resources"] if r["format"].lower() == "csv")

resource = next(r for r in record["resources"] if r["name"] == artifact.name)
response = dataportal.request("GET", resource["url"])
response.raise_for_status()
downloaded_sha256 = hashlib.sha256(response.content).hexdigest()
committed_sha256 = hashlib.sha256(artifact.read_bytes()).hexdigest()

checks = {
    "title": record["title"] == metadata.title,
    "description": record["notes"] == metadata.description,
    "license": record["license_id"] == metadata.license,
    "creators": [creator["name"] for creator in record["creator"]] == [person.name for person in metadata.creators],
    "keywords": {tag["name"] for tag in record["tags"]} == set(metadata.keywords),
    "keywords = PMDco 2 tags + PMDco 3, TTO": {tag["name"] for tag in record["tags"]} == pmdco2_tags | {"PMDco 3", "TTO"},
    "publication date": record["issued"] == metadata.publication_date.isoformat(),
    f"CSV links = csvw:url values of the file ({len(graph_csv_urls)})": record_csv_urls == graph_csv_urls,
    "artifact SHA-256 = committed file": downloaded_sha256 == committed_sha256,
    "artifact SHA-256 = record's sha256 extra": downloaded_sha256 == extras["pmdco3-tto-recombined"]["sha256"],
}
display(pd.Series(checks, name="equal").to_frame())
print("SHA-256 of the artifact:", downloaded_sha256)

## Summary

- PMDco 2 and PMDco 3 describe Zx3 with different structures: values on characteristics with readable class names, against qualities, measurement data and value specifications with numeric class IDs (section 2).
- Each profile keeps the ontology-specific SPARQL and unit aliases. Its conformance ASKs all hold on its own generation and none on the other (section 3).
- The exploration and acquisition nodes give the same specimens, orientations, records and lab values from both generations, and both point to the same curve bytes; only the provenance IRI differs (section 4).
- The PMDco 2 graph links the curve CSVs through CSVW tables. The original PMDco 3 graph links none; the curve tables that this demonstrator added to the PMDco 3 slices link the same files, declare the CSV dialect and repeat the exact headers (section 5).
- The harvester suggested description, keywords and, for PMDco 3, a date from the graphs, with the triples as evidence, and blocked conversion until creators and description were reviewed. The published PMDco 3 record equals the reviewed metadata, and its artifact is the committed file (sections 6 and 7).

Limits, briefly:

- No graph states a license, and the PMDco 2 graph does not pair its creator names with their ORCIDs; both had to be supplied by a person.
- Ontodocker returns `xsd:string` literals as plain literals: the same RDF 1.1 term, but a strict rdflib comparison sees a difference.
- The PMDco 3 curve tables and the relinked lab table are additions by this demonstrator, marked in the data by `rdfs:comment`, and the slice titles carry the orientation. None of this is part of the TTO original, and the TTO maintainers have not reviewed it.
- The profiles carry declared conventions where the graphs are not explicit (see their module docstrings): in PMDco 2, S0 is told apart from Su and s1–s3 only by the IRI; in PMDco 3, the orientation is a generic quality and notes are recognised by an instance label; both use the non-QUDT unit IRIs `unit:MegaPa` and `unit:GigaPa`, and PMDco 2 also `unit:kiloN` for the maximum force.
- In PMDco 3, L0 shares a class with the extensometer's gauge length, and Fm with the load cell's force range, so every quantity has to be constrained by its bearer, as the profile does.
- Questionable terms in the PMDco 3 data: `pmdco:PMD_0020005`, labelled *porosity* (an `owl:Class`) in the pinned ontology files, links the test piece to its material; several TTO classes used by the data, e.g. `tto:TTO_0000059` (upper yield strength) and `tto:TTO_0000022` (lower yield strength), are marked deprecated in the same commit's source ontology (`src/ontology/tto-full.ttl`), though not in the released file.

Next: [notebook 04](04-tensile-workflow-knowledge-graph.ipynb) shows what the workflow itself records as a knowledge graph and joins it with these data graphs; [notebook 05](05-tensile-analysis-overview.ipynb) runs the analysis on all ten specimens from both generations.